# 05 Gold Final Summary

This notebook consolidates the Gold audit outputs into a single readiness view
for model development. It is a reporting asset, not a feature builder.

In [1]:
from pathlib import Path
import json
from datetime import datetime

import numpy as np
import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

NOTEBOOK_NAME = "05_gold_final_summary"
GOLD = ROOT / "data" / "gold"
FEATURES = GOLD / "features"
LABELS = GOLD / "labels"
TRAINING = GOLD / "training"
METADATA = GOLD / "metadata"

OUTPUT_TABLES = ROOT / "eda" / "gold" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "gold" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "gold" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "gold" / "insights"
CHECKPOINTS = ROOT / "eda" / "gold" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(title: str, observations: list[str], issues: list[str], recommendations: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n"
    (INSIGHTS / f"{NOTEBOOK_NAME}.md").write_text(content, encoding="utf-8")

def save_chart(fig, name: str) -> None:
    fig.write_html(OUTPUT_CHARTS / f"{name}.html", include_plotlyjs="cdn")

print("=" * 72)
print(f"GOLD EDA - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Gold root: {GOLD}")


GOLD EDA - 05_gold_final_summary
Start time: 2026-06-02 14:28:58.060916
Gold root: D:\F1_WinRate_Predictor\data\gold


## Gold Readiness Matrix

The Gold layer is ready for model training only when labels, feature coverage,
leakage, and horizon datasets all pass their minimum gates.

In [2]:
label_contract = read_json(METADATA / "label_contract.json")
feature_contract = read_json(METADATA / "feature_contract.json")
leakage = read_json(METADATA / "leakage_report.json")
training_contract = read_json(METADATA / "training_dataset_contract.json")

readiness = pd.DataFrame([
    {"gate": "labels", "metric": "label rows", "value": int(label_contract["rows"]), "status": "PASS" if label_contract["rows"] > 0 else "FAIL"},
    {"gate": "features", "metric": "master feature rows", "value": int(feature_contract["rows"]), "status": "PASS" if feature_contract["rows"] > 0 else "FAIL"},
    {"gate": "leakage", "metric": "blocker count", "value": len(leakage.get("blockers", [])), "status": leakage.get("status", "FAIL")},
    {"gate": "training", "metric": "training datasets", "value": len(training_contract.get("datasets", {})), "status": "PASS" if len(training_contract.get("datasets", {})) == 5 else "FAIL"},
])
readiness.to_csv(OUTPUT_TABLES / "gold_readiness_matrix.csv", index=False)

fig = px.bar(
    readiness,
    x="gate",
    y="value",
    color="status",
    text="value",
    title="Gold Layer Readiness Matrix",
    labels={"gate": "Gold gate", "value": "Metric value"},
)
fig.update_layout(margin=dict(l=10, r=10, t=55, b=80))
fig.update_traces(textposition="outside", cliponaxis=False)
save_chart(fig, "gold_readiness_matrix")
fig.show()

display(readiness)

,gate,metric,value,status
0,labels,label rows,1374,PASS
1,features,master feature rows,63676,PASS
2,leakage,blocker count,0,PASS
3,training,training datasets,5,PASS


## Modeling Boundary Summary

Gold owns feature construction, label isolation, horizon selection, and leakage
metadata. Model training should own split strategy, preprocessing fit,
estimator selection, calibration, and evaluation.

In [3]:
feature_groups = []
for name, contract in feature_contract.get("feature_tables", {}).items():
    feature_groups.append({
        "feature_group": name,
        "artifact": contract.get("artifact"),
        "rows": contract.get("rows"),
        "allowed_columns": len(contract.get("model_allowed_columns", [])),
        "availability_rule": contract.get("availability_rule"),
    })
feature_group_summary = pd.DataFrame(feature_groups)
feature_group_summary.to_csv(OUTPUT_TABLES / "feature_group_summary.csv", index=False)

fig = px.bar(
    feature_group_summary,
    x="feature_group",
    y="allowed_columns",
    text="allowed_columns",
    title="Allowed Model Features by Gold Feature Group",
    labels={"feature_group": "Feature group", "allowed_columns": "Whitelisted columns"},
)
fig.update_layout(margin=dict(l=10, r=10, t=55, b=80))
fig.update_traces(textposition="outside", cliponaxis=False)
save_chart(fig, "allowed_features_by_group")
fig.show()

display(feature_group_summary)

,feature_group,artifact,rows,allowed_columns,availability_rule
0,base,data/gold/features/master_lap_features.parquet,63676,12,Uses only static session/driver context and cu...
1,position,data/gold/features/position_lap_features.parquet,63676,7,Uses latest known position event at or before ...
2,pit,data/gold/features/pit_lap_features.parquet,63676,5,Uses only pit events strictly before the curre...
3,stint,data/gold/features/stint_lap_features.parquet,63676,7,Uses current stint assignment for the current ...
4,interval,data/gold/features/interval_lap_features.parquet,63676,6,Uses the latest interval snapshot strictly bef...
5,weather,data/gold/features/weather_lap_features.parquet,63676,7,Uses the latest session weather observation st...
6,overtake,data/gold/features/overtake_lap_features.parquet,63676,5,"Assigns timestamped overtake events to laps, t..."


In [4]:
overall_status = "PASS" if readiness["status"].eq("PASS").all() else "REVIEW"
write_report("gold_final_summary", {
    "overall_status": overall_status,
    "readiness": readiness.to_dict(orient="records"),
    "feature_groups": feature_group_summary.to_dict(orient="records"),
})
write_insight(
    "Gold Final Summary",
    [
        f"Overall Gold readiness status is {overall_status}.",
        f"Master feature table contains {feature_contract['rows']:,} lap-grain rows.",
        f"Leakage report status is {leakage.get('status')}.",
        "Gold now exposes separate feature, label, and horizon training artifacts.",
    ],
    [] if overall_status == "PASS" else ["At least one Gold readiness gate needs review."],
    [
        "Move next to model training using feature_contract.json as the feature whitelist.",
        "Keep Gold EDA notebooks as audit-only assets; do not rebuild features inside notebooks.",
    ],
)
(CHECKPOINTS / "gold_final_summary_completed.txt").write_text(datetime.now().isoformat(), encoding="utf-8")

26